# שאלה 1 — בעיית ספקים / צרכנים (פתרון תכנית לינארית)

**קורס:** אלגוריתמים מתקדמים למדמ״ח — תשפ״ו, סמסטר אביב

המחברת פותרת את התכנית הלינארית של בעיית הספקים/צרכנים ומחזירה את הכמות
הכוללת המקסימלית שמתקבלת אצל הצרכנים, יחד עם פתרון אופטימלי \(F\).

## ניסוח התכנית הלינארית (LP)

**משתנים:** לכל קשת $(u_i, v_j) \in E$ נגדיר משתנה $x_{ij} \ge 0$ — הכמות שהספק $u_i$ מספק לצרכן $v_j$.

**פונקציית מטרה** (מקסום הכמות הכוללת שמתקבלת אצל הצרכנים):
$$\max \sum_{(u_i, v_j) \in E} x_{ij}$$

**אילוצים:**
$$\sum_{j:\,(u_i,v_j)\in E} x_{ij} \le a_i \qquad \forall\, 1 \le i \le n \quad (\text{קיבולת ספק})$$
$$\sum_{i:\,(u_i,v_j)\in E} x_{ij} \le b_j \qquad \forall\, 1 \le j \le m \quad (\text{קיבולת צרכן})$$
$$x_{ij} \ge 0 \qquad \forall\, (u_i,v_j)\in E$$

### התקנת PuLP (נדרש בקולב)

In [ ]:
!pip install pulp -q

### נתונים מתוך איור 1

In [ ]:
# קיבולת ספקים a_i (כמה כל ספק יכול לספק לכל היותר)
a = {"u1": 3.5, "u2": 2.0, "u3": 4.0, "u4": 1.5}

# קיבולת צרכנים b_j (כמה כל צרכן יכול לצרוך לכל היותר)
b = {"v1": 2.5, "v2": 3.0, "v3": 2.0, "v4": 1.5, "v5": 2.0}

# קשתות הגרף הדו-צדדי E: (ספק, צרכן) אם הספק יכול לספק לצרכן (לפי איור 1)
edges = [
    ("u1", "v1"), ("u1", "v2"), ("u1", "v4"),
    ("u2", "v2"), ("u2", "v5"),                    # u2 מחובר רק ל-v2, v5
    ("u3", "v1"), ("u3", "v3"), ("u3", "v4"),
    ("u4", "v2"), ("u4", "v3"), ("u4", "v5"),
]

### בניית התכנית הלינארית ופתרונה

In [ ]:
import pulp

prob = pulp.LpProblem("Suppliers_Consumers", pulp.LpMaximize)

# משתני החלטה: x[(u, v)] >= 0 לכל קשת
x = {(u, v): pulp.LpVariable(f"x_{u}_{v}", lowBound=0) for (u, v) in edges}

# פונקציית מטרה: מקסום סך הכמות שמתקבלת אצל הצרכנים
prob += pulp.lpSum(x.values()), "Total_delivered"

# אילוץ קיבולת לכל ספק
for u in a:
    prob += pulp.lpSum(x[(uu, v)] for (uu, v) in edges if uu == u) <= a[u], f"supply_{u}"

# אילוץ קיבולת לכל צרכן
for v in b:
    prob += pulp.lpSum(x[(u, vv)] for (u, vv) in edges if vv == v) <= b[v], f"demand_{v}"

prob.solve(pulp.PULP_CBC_CMD(msg=False))

print("Status:", pulp.LpStatus[prob.status])
print("כמות כוללת מקסימלית שמתקבלת אצל הצרכנים =", pulp.value(prob.objective))

### הפתרון האופטימלי F

In [ ]:
print("F (פתרון אופטימלי) — רק קשתות עם כמות חיובית:")
F = []
for (u, v) in edges:
    val = x[(u, v)].value()
    if val and val > 1e-9:
        F.append(((u, v), round(val, 4)))
        print(f"   (({u}, {v}), {round(val, 4)})")

### אימות חוקיות הפתרון

In [ ]:
# בדיקה: אף ספק לא חורג מ-a_i ואף צרכן לא חורג מ-b_j
print("ניצול קיבולת הספקים:")
for u in a:
    used = sum(x[(uu, v)].value() for (uu, v) in edges if uu == u)
    print(f"   {u}: {round(used, 4)} / {a[u]}")

print("ניצול קיבולת הצרכנים:")
for v in b:
    used = sum(x[(u, vv)].value() for (u, vv) in edges if vv == v)
    print(f"   {v}: {round(used, 4)} / {b[v]}")

## פירוש התוצאה

- הערך שהתקבל, **11.0**, הוא הכמות הכוללת המקסימלית שאפשר לספק לצרכנים — התשובה לסעיף (א.ii) "פתרון אופטימלי" ולסעיף (ג).
- כאן כל ספק וכל צרכן מנוצלים במלואם (היצע כולל = ביקוש כולל = 11), כך שהפתרון מרווה את כל הקיבולות.
- קבוצת `F` שמודפסת היא פתרון אופטימלי חוקי. כדי לקבל **פתרון חוקי שאינו אופטימלי** (כנדרש גם בסעיף א.ii) אפשר לבחור כמויות קטנות יותר, למשל $((u_1,v_1),2),((u_2,v_5),2),((u_3,v_3),2)$ שערכו 6.